In [ ]:
from pathlib import Path
import sys

# Locate this package and load the shared v3 LE locations (config/paths.json,
# overridable with the V3LE_ROOT environment variable).
PROJECT_ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (path / "scripts" / "paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start this notebook from inside the polar_analysis package.")

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from paths import V3LE_DATA_DIR, V3LE_DIAG_DIR, V3LE_FIG_ROOT, fig_dir, require

FIG_DIR_ROOT = fig_dir("time_series_imsk")

In [12]:
import os
import re
import json
import numpy as np
import pandas as pd
import xarray as xr
from collections import OrderedDict
from typing import Optional, Tuple, Set
from xarray.coders import CFDatetimeCoder

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.dates import AutoDateLocator, AutoDateFormatter
from collections import OrderedDict
import matplotlib.dates as mdates

# --- Imports
from pathlib import Path    
from functools import partial

from data_utils import (
    REFERENCE_VAR_DICT,
    define_region,
    list_regions, 
    gen_land_mask, 
    gen_ice_mask,
    normalize_longitude,
    get_run_meta,
    list_runs,
    get_var_meta,
    get_levels,
)

In [21]:
class AnnualAnomalyDataProcessor:
    """
    Read ensemble member files, seasonal-reduce to yearly, compute stats,
    and export a compact NetCDF used by the plotting stage.
    """
    def __init__(
        self, 
        data_dir, 
        file_template, 
        num_ens, 
        var_dict,
        var_key="", 
        var_type="anomaly", 
        month_map=None,
        engine="netcdf4", 
        chunks=None
    ):
        self.data_dir = data_dir
        self.file_template = file_template         # e.g. "{group}.en%(ENS).mpas_ts.YYYYMM-YYYYMM.nc"
        self.num_ens = int(num_ens)
        self.var_dict = var_dict or {}
        self.var_key = var_key
        self.var_type = var_type                   # "anomaly" or "raw"
        self.include_raw_when_anomaly = True
        self.engine = engine
        self.chunks = chunks
        self.month_map = month_map
        self.time_coord = None
        self.bad_files = []
        if month_map is None:
            self.month_map = {
                "MAM":   [3, 4, 5],
                "JJA":   [6, 7, 8],
                "SON":   [9, 10, 11],
                "DJF":   [12, 1, 2],
                "Freeze":[10, 11, 12, 1, 2, 3],
                "Melt":  [4, 5, 6, 7, 8, 9],
                "Growth":[10, 11, 12],
                "Decay": [4, 5, 6],
                "JAS":   [7, 8, 9],
                "JFM":   [1, 2, 3],
                "ANN":   list(range(1, 13)),
            }
        # for data from mpas_analysis
        self.mpas_region_dict = {
            "Arctic"           : {"name":  "arctic",     "index": 0},
            "Equator"          : {"name":  "equatorial", "index": 1},
            "Southern Ocean"   : {"name":  "so",         "index": 2},
            "Nino3"            : {"name":  "nino3",      "index": 3},
            "Nino4"            : {"name":  "nino4",      "index": 4},
            "Nino3.4"          : {"name":  "nino3.4",    "index": 5},
            "Global"           : {"name":  "global",     "index": 6}
        }
        # for data from e3sm_diag 
        self.e3sm_region_dict = {
            "Global"               : {"name":  "global", "index": 0},
            "Northern Hemisphere"  : {"name":  "nh",     "index": 1},
            "Southern Hemisphere"  : {"name":  "sh",     "index": 2}
        }
        
    # ---------- naming helpers ----------
    def _has_D(self, s): return s.startswith("D")
    def _strip_D(self, s): return s[1:] if s.startswith("D") else s
    def _maybe_add_D(self, s): return s if s.startswith("D") or self.var_type != "anomaly" else "D" + s

    def _best_base_match(self, s_woD):
        keys = list(self.var_dict.keys())
        cands = [k for k in keys if (s_woD == k) or s_woD.startswith(k + "_")]
        if cands:
            base = max(cands, key=len)
            rest = s_woD[len(base):]
            suffix = rest if (rest and rest.startswith("_")) else (("_" + rest) if rest else "")
            return base, suffix
        parts = s_woD.split("_")
        if len(parts) >= 2:
            base = "_".join(parts[:2]); suffix = "" if len(parts) == 2 else "_" + "_".join(parts[2:])
        else:
            base = s_woD; suffix = ""
        return base, suffix

    def _split_base_suffix(self, name, expect_anom=None):
        if expect_anom is None:
            expect_anom = (self.var_type == "anomaly")
        s_woD = self._strip_D(name) if expect_anom else name
        return self._best_base_match(s_woD)

    def _resolve_varname(self, var_name: str) -> str:
        base_in, suffix_in = self._split_base_suffix(var_name, expect_anom=None)
        suffix = suffix_in or ""
        if self.var_key:
            tail = f"_{self.var_key}"
            if not suffix.endswith(tail):
                suffix = (suffix + tail) if suffix else tail
        name = base_in + suffix
        if self.var_type == "anomaly":
            name = self._maybe_add_D(name)
        else:
            name = self._strip_D(name)
        name = re.sub(r"__+", "_", name)
        name = re.sub(r"^D_", "D", name)
        return name

    def _target_signature(self, var_name):
        expect_anom = (self.var_type == "anomaly")
        base, _ = self._split_base_suffix(self._resolve_varname(var_name), expect_anom=expect_anom)
        req_suffix = f"_{self.var_key}" if self.var_key else ""
        return expect_anom, base, req_suffix

    def _resolve_existing_var(self, ds, var_name):
        expect_anom, base_tgt, req_suffix = self._target_signature(var_name)
        exact, loose = [], []
        for v in ds.data_vars:
            is_anom = v.startswith("D")
            if expect_anom and not is_anom: continue
            if (not expect_anom) and is_anom: continue
            base_v, suffix_v = self._split_base_suffix(v, expect_anom=is_anom)
            if base_v == base_tgt:
                if req_suffix and suffix_v.endswith(req_suffix): exact.append(v)
                else: loose.append(v)
        if exact: return max(exact, key=len)
        if loose:
            cand = max(loose, key=len)
            print(f'[WARN] Using "{cand}" for "{self._resolve_varname(var_name)}" (suffix mismatch).')
            return cand
        return None

    def _scale_for(self, base_name):  # base name (no D)
        return float(self.var_dict.get(base_name, {}).get("scale", 1.0))

    def _safe_decode_time_for_check(self, ds: xr.Dataset) -> xr.Dataset:
        """
        Ensure there is a usable datetime-like 'time' coord for resample/groupby.
        Supports:
          - MPAS char times (e.g., xtime_startMonthly, xtime_start, xtime)
          - Numeric CF times with 'units'[/calendar]
          - Already datetime64 or cftime
        """
        def _make_datetime64_from_char(char_da: xr.DataArray) -> np.ndarray:
            # char array (n, strlen) -> vector of strings -> pandas datetime
            arr = np.asarray(char_da.values)
            # handle bytes/char arrays robustly
            if arr.dtype.kind in ("S", "U"):  # already 1-D string array?
                s = arr.astype(str)
            else:
                s = np.apply_along_axis(lambda r: "".join([c.decode() if isinstance(c, (bytes, np.bytes_)) else str(c) for c in r]), 1, arr)
            return pd.to_datetime(s, errors="coerce").to_numpy()

        def _ensure_unique_sorted(out: xr.Dataset) -> xr.Dataset:
            out = out.sortby("time")
            try:
                vals = np.asarray(out["time"].values)
                _, idx = np.unique(vals, return_index=True)
                if len(idx) != out.sizes["time"]:
                    out = out.isel(time=np.sort(idx))
            except Exception:
                pass
            return out

        # 0) If 'time' missing, try to synthesize from common MPAS/E3SM fields
        if "time" not in ds:
            # MPAS monthly outputs often carry char time stamps
            for cand in ("xtime_startMonthly", "xtime_start", "xtime"):
                if cand in ds:
                    try:
                        tvec = _make_datetime64_from_char(ds[cand])
                        if np.all(pd.isna(tvec)):
                            continue
                        # attach as coord aligned to the same leading dimension
                        time_dim = ds[cand].dims[0]
                        ds = ds.assign_coords(time=(time_dim, tvec))
                        return _ensure_unique_sorted(ds)
                    except Exception:
                        pass
            # CF numeric time variables named something else?
            for cand in ("Time", "timeMonthly_avg", "t", "TIME"):
                if cand in ds:
                    try:
                        da = ds[cand]
                        units = da.attrs.get("units", None)
                        calendar = da.attrs.get("calendar", "standard")
                        if units is not None:
                            import cftime
                            # cftime.num2date -> cftime objects, which xarray .dt supports
                            dt = cftime.num2date(np.asarray(da.values), units=units, calendar=calendar, only_use_cftime_datetimes=True)
                            ds = ds.assign_coords(time=(da.dims[0], np.array(dt, dtype="object")))
                            return _ensure_unique_sorted(ds)
                    except Exception:
                        pass
            # As a last resort, leave as-is (downstream may still work if not seasonal)
            return ds

        # 1) We have a 'time' coord; make sure it is datetime-like or cftime
        t = ds["time"]
        # Already numeric? try CF decode via attrs
        if np.issubdtype(t.dtype, np.number):
            units = t.attrs.get("units", None)
            calendar = t.attrs.get("calendar", "standard")
            if units is not None:
                try:
                    import cftime
                    dt = cftime.num2date(np.asarray(t.values), units=units, calendar=calendar, only_use_cftime_datetimes=True)
                    out = ds.assign_coords(time=(t.dims[0], np.array(dt, dtype="object")))
                    return _ensure_unique_sorted(out)
                except Exception:
                    pass
            # If we can't decode, keep numeric but sorted; seasonal ops needing .dt will fail later
            return _ensure_unique_sorted(ds)

        # String/object → try pandas
        if t.dtype.kind in ("O", "S", "U"):
            try:
                tt = pd.to_datetime(np.asarray(t.values), errors="coerce")
                if np.all(pd.isna(tt)):
                    return _ensure_unique_sorted(ds)
                out = ds.assign_coords(time=(t.dims[0], tt))
                return _ensure_unique_sorted(out)
            except Exception:
                return _ensure_unique_sorted(ds)

        # cftime or datetime64 → just sort & de-dup
        return _ensure_unique_sorted(ds)


    def _preprocess(self, ds, season):
        # Make sure 'time' is decode-able / safe for resample() and does not trip isfinite()
        ds = self._safe_decode_time_for_check(ds)

        bases  = set(self.var_dict.keys())
        req_suffix = f"_{self.var_key}" if self.var_key else ""
        keep = []
        expect_anom = (self.var_type == "anomaly")
        keep_raw_also = expect_anom and getattr(self, "include_raw_when_anomaly", True)

        # Select variables of interest
        for v in ds.data_vars:
            is_anom = v.startswith("D")
            if self.var_type == "raw" and is_anom:
                continue
            if self.var_type == "anomaly" and (not is_anom) and (not keep_raw_also):
                continue
            base, suffix = self._split_base_suffix(v, expect_anom=is_anom)
            if (base in bases) and (suffix.endswith(req_suffix) if req_suffix else True):
                keep.append(v)

        if not keep:
            if not getattr(self, "_warned_no_match", False):
                print("[WARN] _preprocess kept no variables matching var_dict keys / var_type / suffix.")
                self._warned_no_match = True
            return ds[[]]

        ds = ds[keep]

        # Region select + seasonal reduction → yearly
        out_vars = {}
        for v in ds.data_vars:
            is_anom = v.startswith("D")
            base, _ = self._split_base_suffix(v, expect_anom=is_anom)
            da = ds[v]
            da = self._select_region_if_needed(da, base)
            da = self._to_yearly_series(da, season)  # coords: time + year
            out_vars[v] = da

        dso = xr.Dataset(out_vars)

        # ensure 'year' is a coord if present
        if "year" in dso and "year" not in dso.coords:
            dso = dso.set_coords("year")
        return dso

    # ---------- season helpers ----------
    def _to_yearly_series(self, da, season):
        """
        Reduce a monthly DataArray to yearly series for a given season name.
        - ANN uses calendar-year means.
        - DJF uses quarters anchored at Dec and assigns to the year of Jan/Feb.
        - Any name in self.month_map is supported (including cross-year composites like Freeze).
        Returns a DataArray with dims ('time', ...) and coords:
            - time: integer years (one per seasonal mean)
            - year: same integer years (convenience)
        """
        s = season.upper()

        # Annual (calendar-year, Jan start)
        if s == "ANN":
            out = da.resample(time="YS-JAN").mean(keep_attrs=True)
            years = pd.to_numeric(out.time.dt.year.data, errors="coerce")
            m = np.isfinite(years)
            out = out.isel(time=m)
            years = years[m].astype(int)
            return out.assign_coords(year=("time", years))

        # DJF (assign to year of January) — keep only complete DJF (3 months)
        if s == "DJF":
            grp = da.resample(time="QS-DEC")                # seasons start Dec 1 (DJF, MAM, JJA, SON)
            qmean = grp.mean(keep_attrs=True)

            # Count samples per quarter using time stamps (robust to NaNs in data)
            ones = xr.DataArray(
                np.ones(da.time.size, dtype="i1"),
                coords={"time": da.time},
                dims=["time"],
            )
            nsamp = ones.resample(time="QS-DEC").sum()

            # keep only complete DJF quarters
            djf_mean  = qmean.sel(time=qmean.time.dt.season == "DJF")
            djf_nsamp = nsamp.sel(time=nsamp.time.dt.season == "DJF")
            djf_mean  = djf_mean.where(djf_nsamp == 3, drop=True)

            # label by January's calendar year (Dec->year+1)
            years = (djf_mean.time.dt.year + 1).astype(int)
            return djf_mean.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")

        # Custom seasons from month_map
        if s not in self.month_map:
            raise ValueError(f"Unsupported season: {season}")

        months = self.month_map[s]
        wraps = (1 in months) and (12 in months)  # cross-year (e.g., Freeze)

        sel = da.where(da.time.dt.month.isin(months), drop=True)
        if sel.sizes.get("time", 0) == 0:
            # Return an empty like-DA with proper coords if season selects nothing
            empty = sel.isel(time=slice(0, 0))
            return empty.assign_coords(year=("time", np.array([], dtype=int)))

        base_year = sel.time.dt.year
        if wraps:
            season_year = xr.where(sel.time.dt.month >= 10, base_year + 1, base_year)
        else:
            season_year = base_year

        # Group and average across the months of each seasonal year
        g = sel.groupby(season_year).mean("time", keep_attrs=True)  # dimension is name of season_year (often 'season_year' or 'group')

        # --- NEW: robustly find the time-like dimension name after groupby
        if "time" in g.dims:
            tdim = "time"
        else:
            # usually a single dim like 'season_year'/'group'
            # fall back to the first dimension
            tdim = list(g.dims)[0]

        # Extract and sanitize years from that dimension
        if tdim in g.coords:
            years = np.asarray(g.coords[tdim].values)
        else:
            years = np.arange(g.sizes[tdim])

        years = pd.to_numeric(years, errors="coerce")
        years = np.asarray(years, dtype="float64")
        m = np.isfinite(years)

        # Mask along the correct dim
        g = g.isel({tdim: m})
        years = years[m].astype(int)

        # Normalize to 'time' dim and attach 'year' coord for downstream consistency
        if tdim != "time":
            g = g.rename({tdim: "time"})

        g = g.assign_coords(time=("time", years), year=("time", years))
        g.attrs["season_name"] = s
        return g

    def _select_region_if_needed(self, da, base_name, regname='Global'):
        """
        Select a region if requested by variable metadata or the regname fallback.

        Priority:
          1) var_dict[base_name]['regidx'] if present (int or str)
          2) regname argument (str), default 'Global'

        Supports:
          - MPAS dims:   nOceanRegions, nRegion, nRegions  -> self.mpas_region_dict
          - E3SM dims:   region, rgn                       -> self.e3sm_region_dict
          - Label-based selection if the region axis has string labels
        """
        # --- resolve desired region spec (index or name) ---
        meta = self.var_dict.get(base_name, {}) if base_name else {}
        reg_spec = meta.get("regidx", None)
        if reg_spec is None:
            # try typical alternates then fallback to function arg
            reg_spec = meta.get("region", meta.get("region_name", None))
        
        # Nothing to do
        if reg_spec is None:
            return da
        else:
            # diagnostic
            print(f"[region-select] base={base_name:<15} | reg_spec={reg_spec!r:<10}")
        
        # Keep original priority/order to preserve behavior
        region_dim_order = ("nOceanRegions", "region", "rgn", "nRegion", "nRegions")
        rdim = next((d for d in region_dim_order if d in da.dims), None)
        if rdim is None:
            return da  # variable has no region dimension

        mpas_dims = ("nOceanRegions", "nRegion", "nRegions")
        region_map = self.mpas_region_dict if rdim in mpas_dims else self.e3sm_region_dict

        # If the region coordinate has string labels, try direct .sel first when reg_spec is a str
        coord = da.coords.get(rdim, None)
        if isinstance(reg_spec, str) and coord is not None:
            try:
                if str(coord.dtype).startswith(("object", "U", "S")):
                    if reg_spec in coord.values:
                        return da.sel({rdim: reg_spec})
                    # case-insensitive attempt
                    lower_vals = np.char.lower(coord.values.astype(str))
                    where = np.where(lower_vals == reg_spec.lower())[0]
                    if where.size > 0:
                        return da.isel({rdim: int(where[0])})
            except Exception:
                pass  # fall through to name->index mapping

        # Map name -> index via region_map (matches by dict key or dict['name'])
        reg_idx = None
        if isinstance(reg_spec, (int, np.integer)) or (isinstance(reg_spec, str) and reg_spec.isdigit()):
            reg_idx = int(reg_spec)
        else:
            # string -> try key match
            if reg_spec in region_map:
                reg_idx = int(region_map[reg_spec]["index"])
            else:
                # case-insensitive match on key or 'name'
                tgt = reg_spec.lower()
                for k, v in region_map.items():
                    if k.lower() == tgt or v.get("name", "").lower() == tgt:
                        reg_idx = int(v["index"])
                        break

        # As another fallback, look for common aux name coords to resolve string names
        if reg_idx is None and isinstance(reg_spec, str):
            for cname in ("regionNames", "region_name", "names", "name"):
                if cname in da.coords:
                    aux = da.coords[cname]
                    if str(aux.dtype).startswith(("object", "U", "S")):
                        try:
                            pos = np.where(np.char.lower(aux.values.astype(str)) == reg_spec.lower())[0]
                            if pos.size > 0:
                                reg_idx = int(pos[0])
                                break
                        except Exception:
                            pass

        # Final selection by index (with bounds checks)
        if reg_idx is None:
            # Could not resolve; leave unchanged
            return da

        n = da.sizes.get(rdim, 0)
        if not (0 <= reg_idx < n):
            print(f"[WARN] Region index {reg_idx} is out of bounds for '{rdim}' (size {n}). Returning unmodified.")
            return da

        try:
            return da.isel({rdim: reg_idx})
        except Exception:
            return da

    def _apply_scaling(self, ds):
        updates = {}
        for v in ds.data_vars:
            is_anom = v.startswith("D")
            base, _ = self._split_base_suffix(v, expect_anom=is_anom)
            meta = self.var_dict.get(base, {})
            scale = float(meta.get("scale", 1.0))
            da = ds[v] * scale if scale != 1.0 else ds[v]
            new_attrs = dict(da.attrs)
            meta_unit = meta.get("unit", None)
            if meta_unit:
                if not (da.attrs.get("units","").lower()=="degc" and meta_unit.lower() in ("k","kelvin")):
                    new_attrs["units"] = meta_unit
            if scale != 1.0: new_attrs["scale_applied"] = scale
            if new_attrs != da.attrs:
                updates[v] = da.assign_attrs(new_attrs)
        return ds.assign(**updates) if updates else ds

    # ---------- discovery / load ----------
    def _ensemble_paths(self):
        paths = []
        for i in range(0, self.num_ens):
            ens_str = str(i).zfill(2)  # en00..en24
            fname = self.file_template.replace("%(ENS)", ens_str)
            paths.append(os.path.join(self.data_dir, fname))
        paths = [p for p in paths if os.path.isfile(p)]
        if not paths:
            raise FileNotFoundError(f"No ensemble files {self.file_template} at {self.data_dir}.")
        return sorted(paths)

    def _preprocess_monthly(self, ds):
        """
        Keep only variables of interest, decode/synthesize a safe monthly 'time',
        and apply region selection. DOES NOT reduce to yearly/seasonal.
        """
        ds = self._safe_decode_time_for_check(ds)

        bases  = set(self.var_dict.keys())
        req_suffix = f"_{self.var_key}" if self.var_key else ""
        keep = []
        expect_anom = (self.var_type == "anomaly")
        keep_raw_also = expect_anom and getattr(self, "include_raw_when_anomaly", True)

        for v in ds.data_vars:
            is_anom = v.startswith("D")
            if self.var_type == "raw" and is_anom:
                continue
            if self.var_type == "anomaly" and (not is_anom) and (not keep_raw_also):
                continue
            base, suffix = self._split_base_suffix(v, expect_anom=is_anom)
            if (base in bases) and (suffix.endswith(req_suffix) if req_suffix else True):
                keep.append(v)

        if not keep:
            if not getattr(self, "_warned_no_match_m", False):
                print("[WARN] _preprocess_monthly kept no variables.")
                self._warned_no_match_m = True
            return ds[[]]

        ds = ds[keep]

        # Apply region selection where needed, keep monthly cadence
        out_vars = {}
        for v in ds.data_vars:
            is_anom = v.startswith("D")
            base, _ = self._split_base_suffix(v, expect_anom=is_anom)
            da = ds[v]
            da = self._select_region_if_needed(da, base)
            out_vars[v] = da

        dso = xr.Dataset(out_vars)

        # carry over coords if present
        for cname in ("lat", "lon", "area", "region"):
            if cname in ds.coords and cname not in dso.coords:
                dso = dso.assign_coords({cname: ds[cname]})

        # Ensure 'time' exists and is monotonic monthly
        if "time" not in dso:
            raise ValueError("Monthly dataset lacks 'time' coordinate after preprocessing.")
        dso = dso.sortby("time")

        return dso

    def load_ensemble_yearlies(
        self, 
        season= "ANN", 
        decode_times=False, 
        combine="nested",
        concat_dim="ens",
        chunks=None,
        parallel=False,
        lock=False,
        combine_attrs="drop_conflicts",
        **open_kwargs
    ):
        paths = self._ensemble_paths()
        kwargs = dict(
            combine=combine,
            concat_dim=concat_dim,
            preprocess=partial(self._preprocess, season=season),
            chunks=chunks,
            parallel=parallel,
            engine=self.engine,
            lock=lock,
            decode_times=bool(decode_times),
            combine_attrs=combine_attrs,
        )
        kwargs.update(open_kwargs)
        try:
            ds = xr.open_mfdataset(paths, **kwargs)
            ds = self._apply_scaling(ds)
        except Exception as e:
            print(f"[WARN] open_mfdataset failed ({type(e).__name__}: {e}). Retrying serial open/concat…")
            dsets = []
            for p in paths:
                try:
                    d = xr.open_dataset(p, engine=self.engine, decode_times=False)
                    d = self._preprocess(d,season)  # safe time inside
                    dsets.append(d.expand_dims({"ens": [len(dsets)+1]}))
                except Exception as ee:
                    print(f"[SKIP] {p} -> {type(ee).__name__}: {ee}")
                    self.bad_files.append({"path": p, "error": f"{type(ee).__name__}: {ee}"})
            if not dsets:
                raise RuntimeError("All ensemble files failed to open (after fallback).")
            ds = xr.concat(dsets, dim="ens", combine_attrs="drop_conflicts")
            for d in dsets:
                try: d.close()
                except Exception: pass
            ds = self._apply_scaling(ds)

        ds = ds.assign_coords(ens=("ens", np.arange(1, ds.sizes["ens"]+1)))

        if "time" in ds.dims:
            self.time_coord = "time"
        elif "year" in ds.dims:
            self.time_coord = "year"
        else:
            raise ValueError("No 'time' or 'year' dimension after seasonal reduction.")

        ds = ds.sortby(self.time_coord).chunk({"ens": -1, self.time_coord: -1})
        return ds


    def _single_ens_chunk(self, da):
        """Ensure the 'ens' dimension is a single Dask chunk so .quantile works reliably."""
        if "ens" in da.dims:
            arr = getattr(da, "data", None)
            if hasattr(arr, "chunks"):
                axis = da.get_axis_num("ens")
                ch = arr.chunks[axis]
                if len(ch) > 1:
                    return da.chunk({"ens": -1})
        return da

    def _maybe_roll(self, da, rolling=None):
        if rolling and rolling > 1:
            tdim = da.dims[-1]
            return da.rolling({tdim: rolling}, center=True, min_periods=max(1, rolling//2)).mean()
        return da

    # ---------- detrend & stats ----------
    @staticmethod
    def _sliding_detrend_1d(y, x, window: int):
        n = y.size; out = np.full(n, np.nan, dtype=float)
        if window is None or window < 2: return y.astype(float)
        k = window // 2
        for i in range(n):
            i0 = max(0, i - k); i1 = min(n, i + k + 1)
            yy = y[i0:i1]; xx = x[i0:i1]
            m = np.isfinite(yy) & np.isfinite(xx)
            if m.sum() >= 2:
                a, b = np.polyfit(xx[m], yy[m], 1)
                out[i] = y[i] - (a * x[i] + b)
            else:
                out[i] = np.nan
        return out

    def _to_numeric_time(self, tcoord):
        vals = np.asarray(getattr(tcoord, "values", tcoord))
        if np.issubdtype(vals.dtype, np.number): return vals.astype(float)
        try:
            import cftime
            if vals.size > 0 and isinstance(vals.flat[0], cftime.datetime):
                years = np.array([int(t.year) for t in vals], dtype=float)
                doy = np.array([t.timetuple().tm_yday for t in vals], dtype=float)
                # assume NoLeap for seasonal products
                return years + (doy - 1.0) / 365.0
        except Exception:
            pass
        try:
            tt = pd.to_datetime(vals)
            years = tt.year.astype(float)
            doy = tt.dayofyear.astype(float)
            denom = np.where(getattr(tt, "is_leap_year", False), 366.0, 365.0)
            return years + (doy - 1.0) / denom
        except Exception:
            return np.arange(getattr(tcoord, "size", len(vals)), dtype=float)

    def _sliding_detrend(self, da, window: int):
        tdim = da.dims[-1]  # time-like at last in our preprocessing
        x = xr.DataArray(self._to_numeric_time(da[tdim]), dims=(tdim,))
        detr = xr.apply_ufunc(
            self._sliding_detrend_1d, da, x,
            input_core_dims=[[tdim],[tdim]], output_core_dims=[[tdim]],
            vectorize=True, dask="parallelized", output_dtypes=[float],
            kwargs={"window": int(window)}, dask_gufunc_kwargs={"allow_rechunk": False},
        )
        detr = detr.assign_coords({tdim: da[tdim]}).assign_attrs(da.attrs)
        detr.attrs["detrend"] = f"centered {window}-yr sliding linear"
        return detr

    def _compute_stats_for(self, da, *, pct=(5, 95), rolling=None):
        mean = da.mean("ens", skipna=True)
        std  = da.std("ens",  skipna=True, ddof=1 if da.sizes.get("ens", 0) > 1 else 0)
        lo   = da.min("ens",  skipna=True)
        hi   = da.max("ens",  skipna=True)

        q = self._single_ens_chunk(da).quantile([p/100 for p in pct], dim="ens", skipna=True)
        p_lo = q.sel(quantile=pct[0]/100).reset_coords(names="quantile", drop=True)
        p_hi = q.sel(quantile=pct[1]/100).reset_coords(names="quantile", drop=True)

        if rolling and rolling > 1:
            mean = self._maybe_roll(mean, rolling)
            std  = self._maybe_roll(std,  rolling)
            lo   = self._maybe_roll(lo,   rolling)
            hi   = self._maybe_roll(hi,   rolling)
            p_lo = self._maybe_roll(p_lo, rolling)
            p_hi = self._maybe_roll(p_hi, rolling)

        for arr in (mean, std, lo, hi, p_lo, p_hi):
            arr.attrs.update(da.attrs)

        return {
            "mean": mean, "std": std, "min": lo, "max": hi,
            f"p{pct[0]:02d}": p_lo, f"p{pct[1]:02d}": p_hi
        }

    def export_figure_data(
        self, 
        sub_dict, 
        *, 
        out_nc_path, 
        season="ANN",
        pct=(5,95), 
        rolling=None, 
        detrend_window=None,
        decode_times=True,
    ):
        ds = self.load_ensemble_yearlies(
            season=season,
            decode_times=decode_times
        )
        tdim = self.time_coord
        out_vars = {}
        meta_prov = {
            "season": season, "var_type": self.var_type, "rolling": int(rolling) if rolling else None,
            "pct_band": list(pct), "detrend_window": int(detrend_window) if detrend_window else None,
            "time_coord": tdim,
        }

        for vname in sub_dict.keys():
            v_resolved = self._resolve_varname(vname)
            if v_resolved not in ds:
                alt = self._resolve_existing_var(ds, vname)
                if alt is None:
                    print(f"[SKIP] {vname}: not found in dataset."); continue
                v_resolved = alt
            da = ds[v_resolved]

            s = self._compute_stats_for(da, pct=pct, rolling=rolling)
            for k, arr in s.items(): out_vars[f"{vname}_{k}"] = arr

            if detrend_window and detrend_window >= 2:
                da_detr = self._sliding_detrend(da, int(detrend_window))
                s2 = self._compute_stats_for(da_detr, pct=pct, rolling=rolling)
                for k, arr in s2.items(): out_vars[f"{vname}_{k}_detr"] = arr

        if not out_vars:
            raise RuntimeError("No variables exported — nothing matched sub_dict in the dataset.")
        dso = xr.Dataset(out_vars).sortby(tdim)
        dso.attrs["annual_anomaly_plotter"] = json.dumps(meta_prov)
        Path(out_nc_path).parent.mkdir(parents=True, exist_ok=True)
        comp = dict(zlib=True, complevel=4)
        enc = {vn: comp for vn in dso.data_vars}
        dso.to_netcdf(out_nc_path, encoding=enc)
        print(f"[INFO] wrote figure-data NetCDF: {out_nc_path}")
        return out_nc_path


In [27]:
if __name__ == "__main__":
    # --- Paths
    data_dir = f"{V3LE_DATA_DIR}/sea_ice"
    out_dir  = str(V3LE_DIAG_DIR)
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    # --- Ensemble naming / meta
    group   = "v3.LR.historical"
    regnam  = "Arctic"
    nens    = 25
    period  = "1850-2050"
    detrend_window = 10  # yrs
    force_recompute = True  # True to overwrite existing stats files
    pct     = (5, 95)    # percentile band to save alongside mean/std/min/max
    rolling = None       # optional temporal smoothing of saved stats
    engine  = "netcdf4"
    
    assert isinstance(nens, int) and nens > 0, "nens must be a positive integer"

    if detrend_window is not None:
        assert int(detrend_window) >= 3, "Use a detrend window ≥ 3 years for stability"
        
    start_year, end_year = period.split("-")
    date_span = f"{start_year}01-{end_year}12"
    
    #seasons = ["MAM","JJA","SON","DJF","ANN"]
    seasons = None # all possible seasons
    
    # --- Variables (units appear in y-labels)
    var_typ = "raw"   # or "anomaly"  (NOT "anom")
    var_key = ""
    overrides = {
        "iceVolume":    {
            "name": "Ice Volume", 
            "unit": r"10$^3$ km$^3$", 
            "min": -20, "max": 10, 
            "dmin": -4, "dmax": 4, 
            "nlev": 11, "scale": 1e-12},
        "iceThickness": {
            "name": "Ice Thickness", 
            "unit": "mm",          
            "min": -50, "max": 20, 
            "dmin": -10, "dmax": 10, 
            "nlev": 11, "scale": 1e3
        },
        "iceArea":      {
            "name": "Ice Area",     
            "unit": r"10$^6$ km$^2$", 
            "min": -5,  "max": 5,  
            "dmin": -1, "dmax": 1,  
            "nlev": 11, "scale": 1e-12
        },
        "sst":          {
            "name": "Sea Surface Temperature", 
            "unit": "K",  
            "min": -0.1, "max": 0.15, 
            "dmin": -0.05, "dmax": 0.05, 
            "nlev": 11, "scale": 1.0, 
            "region" : regnam
            #"regidx": reg_dict[regnam]["index"]
        },
    }

    # Merge reference dict + overrides (won't crash if reference dict is absent)
    try:
        base_dict = REFERENCE_VAR_DICT.copy()
    except NameError:
        base_dict = {}
    
    var_dict = {**base_dict, **overrides}
    
    # Subset for processing
    sub_keys = ["iceVolume", "iceThickness", "iceArea", "sst"]
    sub_dict = {k: var_dict[k] for k in sub_keys if k in var_dict}
    if not sub_dict:
        raise ValueError("No matching variables found in var_dict for sub_keys!")

    # Template with %(ENS) placeholder
    file_template = f"{group}.en%(ENS).mpas_ts.{date_span}.nc"

    # --- Phase A: produce stats once (per season) ---
    processor = AnnualAnomalyDataProcessor(
        data_dir=data_dir,
        file_template=file_template,
        num_ens=nens,
        var_dict=var_dict,
        var_key=var_key,
        var_type=var_typ,     # "raw" or "anomaly"
        month_map=month_map,
        engine=engine,
    )
    if seasons is None:
        seasons = list(processor.month_map.keys())

    
    for season in seasons:
        if season not in processor.month_map.keys():
            raise ValueError(f"Unexpected season '{season}'")
            
        stats_nc = Path(out_dir) / f"{group}.{regnam}.{season}.ensemble_stats.{date_span}.nc"
        if force_recompute and stats_nc.exists():
            try:
                stats_nc.unlink()
                print(f"[info] Removed existing stats to recompute: {stats_nc}")
            except Exception as e:
                print(f"[warn] Could not remove {stats_nc}: {e}")

        if not stats_nc.is_file():
            processor.export_figure_data(
                season=season,
                sub_dict=sub_dict,
                out_nc_path=str(stats_nc),
                pct=pct,
                rolling=rolling,
                detrend_window=detrend_window,
            )
        else:
            print(f"[skip] Stats already present: {stats_nc}")

[info] Removed existing stats to recompute: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/figure_data/v3.LR.historical.Arctic.MAM.ensemble_stats.185001-205012.nc
[region-select] base=sst             | reg_spec='Arctic'  
[region-select] base=sst             | reg_spec='Arctic'  
[region-select] base=sst             | reg_spec='Arctic'  
[region-select] base=sst             | reg_spec='Arctic'  
[region-select] base=sst             | reg_spec='Arctic'  
[region-select] base=sst             | reg_spec='Arctic'  
[region-select] base=sst             | reg_spec='Arctic'  
[region-select] base=sst             | reg_spec='Arctic'  
[region-select] base=sst             | reg_spec='Arctic'  
[region-select] base=sst             | reg_spec='Arctic'  
[region-select] base=sst             | reg_spec='Arctic'  
[region-select] base=sst             | reg_spec='Arctic'  
[region-select] base=sst             | reg_spec='Arctic'  
[region-select] base=sst             | reg_spec='

[SKIP] /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/data/sea_ice/v3.LR.historical.en09.mpas_ts.185001-205012.nc -> TypeError: Variable 'year': Using a DataArray object to construct a variable is ambiguous, please extract the data using the .data property.
[SKIP] /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/data/sea_ice/v3.LR.historical.en10.mpas_ts.185001-205012.nc -> TypeError: Variable 'year': Using a DataArray object to construct a variable is ambiguous, please extract the data using the .data property.
[SKIP] /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/data/sea_ice/v3.LR.historical.en11.mpas_ts.185001-205012.nc -> TypeError: Variable 'year': Using a DataArray object to construct a variable is ambiguous, please extract the data using the .data property.
[SKIP] /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/data/sea_ice/v3.LR.historical.en12.mpas_ts.185001-205012.nc -> TypeError: Variable 'year

RuntimeError: All ensemble files failed to open (after fallback).